# Success Factors Analysis

This notebook consolidates the existing **probing** and **setup-factor** analyses into one entry point.

Corrected names used here:

- `probiong` ? `probing`
- `succsses_factors` / `sucssess` ? `success_factors`

The goal is to collect evidence for factors that may influence trial success, while reusing the existing analysis modules and generated CSV outputs instead of duplicating their logic.


## Factor inventory

Potentially relevant success factors included in this notebook:

1. **Setup factor**: `L` = with airsled, `N` = without airsled.
2. **Participant / subject**: subject-level variability and participant-normalized summaries.
3. **Experimental scope/group**: e.g. `L_E`, `N_E`, filtered vs non-filtered probing scopes when available.
4. **Stiffness / motor condition**: motor-set or stiffness level in trial and summary tables.
5. **Finger / stimulated object**: finger/object role where output tables expose it.
6. **Probing direction relationship**: same, ambiguous, or different standard-vs-comparison movement directions.
7. **Angular direction difference**: continuous direction gap between standard and comparison object trajectories.
8. **8-way movement direction**: E, NE, N, NW, W, SW, S, SE for standard and comparison roles.
9. **Comparison-object direction**: emphasized because participant choice depends on judging the comparison object.
10. **Order / fatigue / response drift**: trial order effects and response drift if probing diagnostics are available.
11. **Success-rate uncertainty**: CIs, trial counts, and low-power setup-balance warnings.

Treat this notebook as an exploratory consolidation. Confirmatory claims should use participant-normalized tables and model/status outputs rather than raw pooled rates alone.


## Setup and imports


In [ ]:

from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    """Find the Parallel_Heptics project root from a notebook or script cwd."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "analysis").exists():
            return candidate
    raise RuntimeError("Could not find project root containing pyproject.toml and analysis/.")


PROJECT_ROOT = find_project_root()
ANALYSIS_ROOT = PROJECT_ROOT / "analysis"
SUCCESS_FACTORS_ROOT = ANALYSIS_ROOT / "success_factors"
PROBING_ROOT = SUCCESS_FACTORS_ROOT
SETUP_FACTOR_ROOT = SUCCESS_FACTORS_ROOT
PROBING_RESULTS_ROOT = SUCCESS_FACTORS_ROOT / "results" / "probing"
SETUP_FACTORS_RESULTS_ROOT = SUCCESS_FACTORS_ROOT / "results" / "setup_factors"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analysis.success_factors import probing  # noqa: E402
from analysis.success_factors import setup_factors as setup_factor  # noqa: E402

print(f"Project root: {PROJECT_ROOT}")
print(f"Probing module: {PROBING_ROOT / 'probing.py'}")
print(f"Setup-factor module: {SETUP_FACTOR_ROOT / 'setup_factors.py'}")
print(f"Success-factors notebook folder: {SUCCESS_FACTORS_ROOT}")


## Small helpers for generated outputs


In [ ]:

def csv_files(root: Path) -> list[Path]:
    """Return generated CSV files under a root, sorted by name."""
    if not root.exists():
        return []
    return sorted(path for path in root.rglob("*.csv") if path.is_file())


def read_csv_if_exists(path: Path, **kwargs) -> pd.DataFrame:
    """Read a CSV when present; return an empty DataFrame when missing."""
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def summarize_csv_inventory(root: Path, label: str) -> pd.DataFrame:
    """Create a compact inventory of generated CSV outputs."""
    rows = []
    for path in csv_files(root):
        rows.append(
            {
                "source": label,
                "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
                "file_name": path.name,
                "size_kb": round(path.stat().st_size / 1024, 1),
            }
        )
    return pd.DataFrame(rows)


setup_inventory = summarize_csv_inventory(SETUP_FACTORS_RESULTS_ROOT, "setup_factor")
probing_inventory = summarize_csv_inventory(PROBING_RESULTS_ROOT, "probing")
inventory = pd.concat([setup_inventory, probing_inventory], ignore_index=True)

print(f"Found {len(setup_inventory)} setup-factor CSVs and {len(probing_inventory)} probing CSVs.")
inventory.head(20)


## Optional regeneration commands

By default, this notebook reads existing generated outputs. If inputs change, uncomment and run the relevant lines below.

> Regeneration can take longer than the summary cells because it may traverse many result folders and write figures/CSVs.


In [ ]:
# Regenerate center-to-side probing outputs (counts, jerk, straightness).
# This cell runs BEFORE the inventory/inspection cells so Run All creates fresh outputs.
RUN_CENTER_TO_SIDE_PROBING = True

# All _E kinematics export: combined L_E + N_E, excluding non-E/pilot protocol groups.
# The folder must contain trial_kinematic_summary.csv and kinematic_samples.parquet/csv.
KINEMATICS_RESULTS_SOURCE = PROJECT_ROOT / "analysis" / "Kinematics" / "results_old2" / "L_N_E" / "csv" / "other"
PROBING_CENTER_TO_SIDE_OUTPUT_ROOT = PROBING_RESULTS_ROOT / "L_N_E_center_to_side"

if RUN_CENTER_TO_SIDE_PROBING:
    if not KINEMATICS_RESULTS_SOURCE.exists():
        raise FileNotFoundError(f"Kinematics source does not exist: {KINEMATICS_RESULTS_SOURCE}")
    print("Running center-to-side probing analysis on all _E kinematics data...")
    print(f"Input:  {KINEMATICS_RESULTS_SOURCE}")
    print(f"Output: {PROBING_CENTER_TO_SIDE_OUTPUT_ROOT}")
    probing_tables = probing.run_analysis(
        KINEMATICS_RESULTS_SOURCE,
        PROBING_CENTER_TO_SIDE_OUTPUT_ROOT,
        fig_dpi=120,
    )
    print("Saved probing outputs:")
    for name, df in probing_tables.items():
        if isinstance(df, pd.DataFrame):
            print(f"  {name}: {df.shape}")
    print(f"CSV count now: {len(csv_files(PROBING_CENTER_TO_SIDE_OUTPUT_ROOT))}")
else:
    print("RUN_CENTER_TO_SIDE_PROBING is False; existing results are only inspected, not regenerated.")

# Optional heavier direction-success batch regeneration. Leave False unless needed.
RUN_DIRECTION_SUCCESS_BATCH = False
if RUN_DIRECTION_SUCCESS_BATCH:
    batch = probing.run_direction_success_batch(PROJECT_ROOT / "results")
    print(batch["batch_manifest"].to_string(index=False))


## Setup-factor diagnostics

These tables summarize whether `L` (with airsled) and `N` (without airsled) are balanced enough to discuss setup as a factor. Low-power flags should be treated as a limitation, not proof of no effect.


In [ ]:

setup_results = SETUP_FACTORS_RESULTS_ROOT
setup_balance_files = sorted(setup_results.glob("*_setup_balance.csv"))
setup_status_files = sorted(setup_results.glob("*_setup_status.csv"))
setup_comparison_files = sorted(setup_results.glob("*_between_setup_metric_comparisons.csv"))

setup_balance = pd.concat(
    [read_csv_if_exists(path).assign(source_file=path.name) for path in setup_balance_files],
    ignore_index=True,
) if setup_balance_files else pd.DataFrame()

setup_status = pd.concat(
    [read_csv_if_exists(path).assign(source_file=path.name) for path in setup_status_files],
    ignore_index=True,
) if setup_status_files else pd.DataFrame()

setup_comparisons = pd.concat(
    [read_csv_if_exists(path).assign(source_file=path.name) for path in setup_comparison_files],
    ignore_index=True,
) if setup_comparison_files else pd.DataFrame()

print("Setup balance rows:", len(setup_balance))
print("Setup status rows:", len(setup_status))
print("Setup comparison rows:", len(setup_comparisons))
setup_balance.head(10)


In [ ]:

if not setup_status.empty:
    display_columns = [column for column in setup_status.columns if column in {"dataset", "status", "message", "source_file"}]
    if not display_columns:
        display_columns = list(setup_status.columns[:8])
    setup_status[display_columns].head(20)
else:
    print("No setup status CSVs were found. Run setup_factor.run_setup_factor_diagnostics() if regeneration is needed.")


## Probing direction-success diagnostics

The probing analysis already exposes trial-level and participant-normalized tables for direction success. The most relevant outputs are:

- same/different direction success summaries;
- comparison-object 8-way direction success;
- standard-object 8-way direction success;
- participant x stiffness x direction summaries;
- paired participant direction contrasts;
- optional logistic/model status tables.


In [ ]:

probing_results = PROBING_RESULTS_ROOT

probing_focus_patterns = [
    "*direction_success*.csv",
    "*same_vs_different*.csv",
    "*participant_x_stiffness_x_direction*.csv",
    "*subject_bias_vs_difference*.csv",
    "*logistic_status.csv",
    "*order_effects*.csv",
    "*probe_metrics_summary.csv",
    "figure_manifest.csv",
]

probing_focus_files: list[Path] = []
for pattern in probing_focus_patterns:
    probing_focus_files.extend(sorted(probing_results.rglob(pattern)))
probing_focus_files = sorted(set(probing_focus_files))

probing_focus_inventory = pd.DataFrame(
    [
        {
            "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
            "file_name": path.name,
            "size_kb": round(path.stat().st_size / 1024, 1),
        }
        for path in probing_focus_files
    ]
)

print(f"Found {len(probing_focus_files)} probing success/direction/order CSVs.")
probing_focus_inventory.head(30)


In [ ]:
def load_first_matching(pattern: str) -> pd.DataFrame:
    """Load the first matching probing results CSV for quick inspection."""
    search_roots = []
    if "PROBING_CENTER_TO_SIDE_OUTPUT_ROOT" in globals() and PROBING_CENTER_TO_SIDE_OUTPUT_ROOT.exists():
        search_roots.append(PROBING_CENTER_TO_SIDE_OUTPUT_ROOT)
    search_roots.append(probing_results)
    matches = []
    for root in search_roots:
        matches.extend(sorted(root.rglob(pattern)))
    matches = sorted(set(matches))
    if not matches:
        print(f"No matches for {pattern}")
        return pd.DataFrame()
    path = matches[0]
    print(f"Loaded {path.relative_to(PROJECT_ROOT)}")
    return pd.read_csv(path)


same_different_success = load_first_matching("*perception_action_direction_success_by_direction.csv")
comparison_direction_success = load_first_matching("*perception_action_success_by_direction.csv")
participant_stiffness_direction = load_first_matching("*participant_x_stiffness_x_direction_success.csv")

subject_probe_metrics = load_first_matching("*subject_finger_stiffness_probe_metrics_summary.csv")
group_probe_metrics = load_first_matching("*group_finger_stiffness_probe_metrics_summary.csv")
analysis_scope_probe_metrics = load_first_matching("*analysis_scope_finger_stiffness_probe_metrics_summary.csv")

same_different_success.head(10)


## Combined success-factor evidence table


In [ ]:

def table_presence_row(name: str, frame: pd.DataFrame, relevant_factors: str) -> dict[str, object]:
    return {
        "evidence_table": name,
        "rows": len(frame),
        "columns": ", ".join(map(str, frame.columns[:12])) if not frame.empty else "",
        "relevant_factors": relevant_factors,
        "available": not frame.empty,
    }


success_factor_evidence = pd.DataFrame(
    [
        table_presence_row(
            "setup_balance",
            setup_balance,
            "setup factor, subject counts, low-power/balance limitation",
        ),
        table_presence_row(
            "setup_status",
            setup_status,
            "setup factor, diagnostic status, whether setup can be interpreted",
        ),
        table_presence_row(
            "setup_comparisons",
            setup_comparisons,
            "setup factor, metric, condition/scope, between-setup differences",
        ),
        table_presence_row(
            "same_different_success",
            same_different_success,
            "same/different probing direction, angular relationship, success rate",
        ),
        table_presence_row(
            "comparison_direction_success",
            comparison_direction_success,
            "comparison-object 8-way direction, success rate",
        ),
        table_presence_row(
            "participant_stiffness_direction",
            participant_stiffness_direction,
            "participant, stiffness, standard/comparison direction, success rate",
        ),
    ]
)

success_factor_evidence


## Interpretation guide

Use this checklist when deciding which factors influence success:

- **Start with participant-normalized summaries** when comparing directions, stiffness, or setup. Raw trial pooling can over-weight subjects with more usable trials.
- **Check setup balance first**. If `L`/`N` counts are low or unbalanced, describe setup as a limitation and avoid strong causal claims.
- **Compare same-vs-different direction** before 8-way direction claims. The binary relationship has clearer interpretation and usually more data per cell.
- **Use comparison-object direction cautiously but prominently** because the participant's response is about whether the comparison object felt stronger.
- **Inspect stiffness/finger interactions** before claiming a general direction effect; a direction may matter only at certain stiffness levels or fingers.
- **Review order-effect diagnostics** if success changes across trial order; apparent factor effects may partly reflect fatigue, learning, or drift.
- **Prefer confidence intervals and counts** over only point estimates, especially for sparse directions such as diagonals.


## Next steps

1. Run all cells top-to-bottom after upstream probing/setup analyses are refreshed.
2. Select the highest-value tables for thesis figures: setup balance/status, same-vs-different direction success, comparison-direction participant-normalized success, and participant x stiffness x direction summaries.
3. If setup is underpowered, report it as a limitation and avoid treating non-significant setup comparisons as proof that setup has no effect.
4. For final claims, pair this notebook with the original generated CSVs and figures under `analysis/success_factors/results/probing` and `analysis/success_factors/results/setup_factors`.
